# Analyse de la baseline autoencodeur

On entraîne un petit autoencodeur, on observe la courbe de perte, la
distribution des scores normal vs anomalie, on choisit un seuil sur la
validation et on lit les métriques. Repli automatique sur des données
**synthétiques** si MVTec n'est pas téléchargé.

> Sur données synthétiques (bruit), les métriques sont volontairement
> médiocres : le but est d'illustrer la démarche, pas la performance.


In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import torch
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

from anomaly_detection.data.synthetic import generate_synthetic_mvtec
from anomaly_detection.data.splits import create_splits
from anomaly_detection.data.datasets import MVTecDataset
from anomaly_detection.models.autoencoder import build_autoencoder
from anomaly_detection.training.engine import train_autoencoder
from anomaly_detection.evaluation.collect import collect_outputs
from anomaly_detection.evaluation.thresholds import threshold_from_normal_percentile
from anomaly_detection.evaluation.metrics import image_level_metrics
from anomaly_detection.inference.postprocessing import min_max_normalize

In [ ]:
# Données : vrai MVTec si présent, sinon synthétique.
DATA = ROOT / "data" / "raw" / "mvtec_ad"
CATEGORY, SIZE = "bottle", 64
if not (DATA / CATEGORY / "train" / "good").is_dir():
    print("MVTec absent -> categorie synthetique")
    generate_synthetic_mvtec(DATA, category="synthetic", n_train_good=16)
    CATEGORY = "synthetic"

splits = create_splits(DATA, CATEGORY, seed=42)
train_ds = MVTecDataset(DATA, CATEGORY, "train", SIZE, splits=splits)
val_ds = MVTecDataset(DATA, CATEGORY, "val", SIZE, splits=splits)
test_ds = MVTecDataset(DATA, CATEGORY, "test", SIZE, splits=splits)
print(f"train={len(train_ds)} val={len(val_ds)} test={len(test_ds)}")

In [ ]:
# Entraînement rapide (CPU) de l'autoencodeur.
device = torch.device("cpu")
model = build_autoencoder({"base_channels": 16, "latent_dim": 32})
history = train_autoencoder(
    model,
    DataLoader(train_ds, batch_size=4, shuffle=True, drop_last=True),
    DataLoader(val_ds, batch_size=4),
    epochs=5,
    learning_rate=1e-3,
    device=device,
    checkpoint_path=ROOT / "models" / "nb_autoencoder.pt",
    seed=42,
    extra={
        "model": "autoencoder",
        "model_config": {"base_channels": 16, "latent_dim": 32},
        "image_size": SIZE,
    },
)
plt.plot(history.train, marker="o", label="train")
plt.plot(history.val, marker="o", label="val")
plt.xlabel("epoque")
plt.ylabel("MSE")
plt.legend()
plt.title("Courbe de perte")
plt.show()

In [ ]:
# Distribution des scores : normal vs anomalie.
val = collect_outputs(model, DataLoader(val_ds, batch_size=4), device)
test = collect_outputs(model, DataLoader(test_ds, batch_size=4), device)
labels, scores = test["labels"], test["scores"]
plt.hist(scores[labels == 0], bins=15, alpha=0.6, label="normal")
plt.hist(scores[labels == 1], bins=15, alpha=0.6, label="anomalie")
plt.xlabel("score d'anomalie")
plt.legend()
plt.title("Distribution des scores")
plt.show()

In [ ]:
# Seuil (percentile des normales de validation) + métriques.
thr = threshold_from_normal_percentile(val["scores"], 95.0)
m = image_level_metrics(labels, scores, thr)
print(f"seuil = {thr:.4f}")
for k in ("auroc", "auprc", "precision", "recall", "f1", "fp", "fn"):
    print(f"{k:10s} {m[k]}")

In [ ]:
# Exemple de carte de chaleur sur l'image la plus suspecte.
i = int(np.argmax(scores))
fig, ax = plt.subplots(1, 2, figsize=(8, 4))
ax[0].imshow(plt.imread(test["paths"][i]))
ax[0].set_title("image")
ax[0].axis("off")
ax[1].imshow(min_max_normalize(test["maps"][i]), cmap="jet")
ax[1].set_title("heatmap")
ax[1].axis("off")
plt.show()

## Conclusions

- La courbe de perte doit décroître ; sinon, augmenter les époques ou le
  learning rate.
- On espère des scores d'anomalie plus élevés pour les images anormales.
- Limite connue : un autoencodeur peut reconstruire certaines anomalies
  (score sous-estimé) — d'où la comparaison avec PatchCore.
- Les vrais chiffres s'obtiennent sur MVTec via Colab GPU.
